# Stage-Two PM Optimization
This notebook demonstrates how to use DESC to perform permanent magnet (PM) optimization assuming they are ideal point dipoles. We will find a set of optimized PMs for the MUSE QA equilibrium (Qian et al. 2023), cold-starting from a fully populated PM grid and then turning off/radially rotating/discretizing the dipoles as needed. For comparisom and benchmarking, we will use the same set of TF coils found by the MUSE paper. Once optimized, the normal field error will be assessed, and for the vacuum equilibrium, field line tracing will be performed to compare the Poincare trace to the equilibrium flux surfaces as well as the MUSE results.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))

In [ ]:
# from desc import set_device
# set_device("gpu")

In [ ]:
import jax

jax.config.update("jax_compilation_cache_dir", "../jax-caches")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

In [1]:
import numpy as np
import jax.numpy as jnp
from desc.backend import print_backend_info
from desc.equilibrium import Equilibrium
from desc.dipole import _Dipole, DipoleSet, import_dipoles, export_dipoles
from desc.magnetic_fields import SumMagneticField, SplineMagneticField
from desc.io import load
from desc.compute import get_params
from desc.plotting import plot_2d, plot_3d, plot_coils, plot_comparison, plot_section, plot_dipoles
from desc.grid import LinearGrid, Grid

from matplotlib.ticker import ScalarFormatter

from desc.objectives import (ObjectiveFunction, 
                             DipoleDiscreteness, 
                             DipoleVolume, 
                             QuadraticFluxPM, 
                             )
from desc.optimize import Optimizer
from desc.integrals import compute_B_plasma
import time
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio

# This ensures Plotly output works in multiple places:
# plotly_mimetype: VS Code notebook UI
# notebook: "Jupyter: Export to HTML" command in VS Code
# See https://plotly.com/python/renderers/#multiple-renderers
pio.renderers.default = "plotly_mimetype+notebook"

print_backend_info()

DESC version=0.17.3+149.g5e897185de.dirty.
Using JAX backend: jax version=0.7.2, jaxlib version=0.7.2, dtype=float64.
Using device: CPU, with 6.00 GB available memory.



# Permanent Magnet and Coil Optimization Metrics

Three primary metrics are used to evaluate the optimized permanent-magnet and coil configuration: **normalized normal magnetic field error**, **dipole discreteness**, and **dipole volume**. Poincaré plots provide an additional visual assessment of the resulting magnetic flux surfaces.

### 1. Normalized Normal Magnetic Field Error

The normal magnetic field error measures how closely the combined magnetic field follows the last closed flux surface (LCFS) of the target equilibrium. Using local normalization by the coil magnetic field magnitude, the average error is

$$
\boxed{
\epsilon_B =
\left\langle
\frac{|\mathbf{B}_{\mathrm{total}}\cdot\hat{\mathbf{n}}|}
{|\mathbf{B}_{\mathrm{coils}}|}
\right\rangle
}
$$

where $\hat{\mathbf{n}}$ is the unit normal vector to the LCFS, and

$$
\mathbf{B}_{\mathrm{total}} =
\mathbf{B}_{\mathrm{coils}} +
\mathbf{B}_{\mathrm{PM}} +
\mathbf{B}_{\mathrm{plasma}}.
$$

For vacuum equilibria, the plasma-current contribution is zero. Smaller values indicate better agreement with the target plasma boundary.

### 2. Dipole Discreteness (dpbin)

The dipole discreteness metric encourages the normalized dipole strengths $\tilde{\rho}_i = \tanh(\rho_i)$ to approach $-1$, $0$, or $+1$, corresponding to reversed, absent, or aligned permanent magnets.

The discreteness residual is defined as

$$
\boxed{
D_{\mathrm{bin}} =
\sqrt{
\sum_{i=1}^{N}
\left[
|\tilde{\rho}_i|(1-|\tilde{\rho}_i|)
\right]^2
}
}
$$

This quantity vanishes when every dipole strength belongs to $\{-1,0,+1\}$. The squared residual is the corresponding unweighted least-squares penalty.

### 3. Dipole Volume (dpvol)

The dipole volume objective penalizes the magnitude of the normalized dipole strengths to encourage configurations requiring less magnetic material.

In the current implementation, the residual for dipole $i$ is

$$
v_i = (\tilde{\rho}_i^2+\varepsilon^2)^{1/4},
\qquad \varepsilon = 10^{-6}.
$$

The corresponding unweighted sum-of-squares penalty is

$$
\boxed{
D_{\mathrm{vol}} =
\sum_{i=1}^{N}
\sqrt{\tilde{\rho}_i^2+\varepsilon^2}
}
$$

This approximates $\sum_i |\tilde{\rho}_i|$. For identical candidate magnets, this provides a dimensionless proxy for total magnetic material usage, rather than a direct physical volume measurement.

### 4. Poincaré Plots

Poincaré plots visualize the trajectories of magnetic field lines generated by the combined coil and permanent-magnet fields. Well-defined nested flux surfaces indicate good magnetic confinement, while islands or stochastic regions reveal deviations from the desired magnetic geometry.

The following functions calculate these metrics and visualize the magnetic field configuration.


In [2]:
def calculate_bn_error(
    eq,
    dipoles,
    coils,
    M=64,
    N=64,
    chunk_size=8,
    plot=True,
):
    """
    Calculate normalized normal magnetic field error on the LCFS.

    Parameters
    ----------
    eq : Equilibrium
        Equilibrium defining the plasma boundary.
    dipoles : DipoleSet
        Permanent magnets producing the dipole field.
    coils : CoilSet
        Coils producing the background magnetic field.
    M, N : int
        Poloidal and toroidal grid resolutions.
    chunk_size : int
        Chunk size for magnetic field calculations.
    plot : bool
        Whether to plot normalized Bn error.

    Returns
    -------
    results : dict
        Normalized Bn error, field values, and summary statistics.
    """

    # Surface grid
    grid = LinearGrid(
        rho=1.0,
        M=M,
        N=N,
        NFP=eq.NFP,
        sym=False,
        endpoint=False,
    )

    # Plasma surface coordinates and unit normals (Cartesian)
    data = eq.compute(["x", "n_rho"], grid=grid, basis="xyz")

    xyz = np.asarray(data["x"])
    normals = np.asarray(data["n_rho"])
    normals = normals/(np.linalg.norm(normals, axis=1, keepdims=True))

    # Coil magnetic field
    B_coils = np.asarray(
        coils.compute_magnetic_field(
            xyz,
            basis="xyz",
            chunk_size=chunk_size,
        )
    )

    # Permanent magnet magnetic field
    B_pm = np.asarray(
        dipoles.compute_magnetic_field(
            xyz,
            basis="xyz",
            chunk_size=chunk_size,
        )
    )

    # Total normal magnetic field
    B_total = B_coils + B_pm
    Bn = np.sum(B_total * normals, axis=1)

    # Local normalization by coil magnetic field magnitude
    B_coils_mag = np.linalg.norm(B_coils, axis=1)

    if np.any(B_coils_mag == 0):
        raise ValueError("Coil magnetic field is zero at some surface points.")

    normalized_Bn = Bn / B_coils_mag

    # Summary statistics
    mean_error = np.mean(np.abs(normalized_Bn))
    rms_error = np.sqrt(np.mean(normalized_Bn**2))
    max_error = np.max(np.abs(normalized_Bn))

    print("========== NORMALIZED BN ERROR ==========")
    print(f"Mean |Bn/Bcoils|: {mean_error:.6%}")
    print(f"RMS  |Bn/Bcoils|: {rms_error:.6%}")
    print(f"Max  |Bn/Bcoils|: {max_error:.6%}")

    # Reshape using grid node coordinates
    nodes = np.asarray(grid.nodes)

    theta, i = np.unique(nodes[:, 1], return_inverse=True)
    phi, j = np.unique(nodes[:, 2], return_inverse=True)

    error_2d = np.empty((len(theta), len(phi)))
    error_2d[i, j] = normalized_Bn

    if plot:
        vmax = np.max(np.abs(error_2d))
        levels = np.linspace(-vmax, vmax, 51)

        fig, ax = plt.subplots(figsize=(10, 6))

        cf = ax.contourf(
            phi,
            theta,
            error_2d,
            levels=levels,
            cmap="RdBu_r",
            extend="both",
        )

        cbar = fig.colorbar(cf, ax=ax, pad=0.03)
        cbar.set_label(
            r"$\mathbf{B}\cdot\hat{\mathbf{n}}/|\mathbf{B}_{\rm coils}|$",
            fontsize=22,
        )

        formatter = ScalarFormatter(useMathText=True)
        formatter.set_scientific(True)
        formatter.set_powerlimits((0, 0))

        cbar.formatter = formatter
        cbar.update_ticks()
        cbar.ax.tick_params(labelsize=15)
        cbar.ax.yaxis.get_offset_text().set_fontsize(14)

        ax.set_xlabel(r"Toroidal angle $\phi$", fontsize=20)
        ax.set_ylabel(r"Poloidal angle $\theta$", fontsize=20)
        ax.tick_params(labelsize=15)

        ax.set_xlim(phi.min(), phi.max())
        ax.set_ylim(0, 2 * np.pi)

        fig.tight_layout()
        plt.show()

    return {
        "grid": grid,
        "Bn": Bn,
        "normalized_Bn": normalized_Bn,
        "error_2d": error_2d,
        "B_coils": B_coils,
        "B_pm": B_pm,
        "mean_error": mean_error,
        "rms_error": rms_error,
        "max_error": max_error,
    }


def calculate_dpbin(dipoles):
    """Calculate DipoleDiscreteness's raw residual and squared penalty."""
    rho = np.asarray(dipoles.rho_tilde, dtype=float)

    d = np.abs(rho) * (1.0 - np.abs(rho))

    dpbin = np.sqrt(np.sum(d**2))

    print(f"Discreteness residual (dpbin): {dpbin:.6e}")
    print(f"Discreteness sum of squares:   {dpbin**2:.6e}")

    return dpbin


def calculate_dpvol(dipoles, eps=1e-6):
    """Calculate DipoleVolume's raw residuals and squared penalty."""
    rho = np.asarray(dipoles.rho_tilde, dtype=float)

    v = (rho**2 + eps**2)**0.25

    dpvol = np.sum(v**2)

    print(f"Dipole volume penalty (dpvol): {dpvol:.6e}")
    print(f"Approximate sum |rho_tilde|:   {np.sum(np.abs(rho)):.6e}")

    return dpvol

In [3]:
muse_eq = Equilibrium.from_input_file(
    "input_output_files/input.muse-fixedb"
)
muse_coilset = load('tf_coils_desc.h5')
muse_dipoles = import_dipoles(
    NFP=2,
    sym=True,
    filename="input_output_files/muse-magnets.focus",
)

Converting VMEC input to DESC input


/Users/jessicali/DESC/desc/equilibrium/initial_guess.py:231: UserWarning:

Surfaces from initial guess are not nested, attempting to refine coordinates. This may take a few moments.

/Users/jessicali/DESC/desc/objectives/utils.py:508: RuntimeWarning:

divide by zero encountered in matmul

/Users/jessicali/DESC/desc/objectives/utils.py:508: RuntimeWarning:

overflow encountered in matmul

/Users/jessicali/DESC/desc/objectives/utils.py:508: RuntimeWarning:

invalid value encountered in matmul

/Users/jessicali/DESC/desc/compat.py:39: UserWarning:

Left handed coordinates detected, switching sign of theta. To avoid this warning in the future, switch the sign of all modes with m<0 and iota/current profile.



In [4]:
# plot rho distribution

import numpy as np
import matplotlib.pyplot as plt

def plot_rho_distribution(dipoles, name="MUSE", bins=80):
    """
    Plot the distribution of normalized dipole strengths.

    Parameters
    ----------
    dipoles : DipoleSet
        Dipole configuration to analyze.
    name : str
        Configuration name for labeling and saving.
    bins : int
        Number of histogram bins.
    """
    rho = np.asarray(dipoles.rho_tilde)

    print(f"========== {name} ==========")
    print(f"Number of dipoles: {len(rho):,}")
    print(f"Mean |rho_tilde|:  {np.mean(np.abs(rho)):.6f}")
    print(f"L2 norm:           {np.linalg.norm(rho):.6f}")
    print(f"Near zero (<0.05): {np.mean(np.abs(rho) < 0.05):.2%}")
    print(f"Near ±1 (>0.95):   {np.mean(np.abs(rho) > 0.95):.2%}")
    print(f"Active (>0.5):     {np.mean(np.abs(rho) > 0.5):.2%}")

    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.hist(
        rho,
        bins=np.linspace(-1, 1, bins + 1),
        color="tab:blue" if name == "QI" else "tab:red",
        edgecolor="white",
        linewidth=0.3,
        alpha=0.85,
    )

    ax.set_xlabel(r"Normalized dipole strength $\tilde{\rho}$", fontsize=20)
    ax.set_ylabel("Number of dipoles", fontsize=20)
    ax.tick_params(labelsize=16)
    ax.set_xlim(-1, 1)

    fig.tight_layout()
    plt.show()

    return rho

In [ ]:
# plot MUSE solution

grid = LinearGrid(
    rho=1.0,
    M=40,
    N=80,
    NFP=1,
    sym=False,
    endpoint=True,
)

fig = plot_3d(muse_eq, "|B|", grid=grid)

# Remove surface colorbar
fig.update_traces(
    showscale=False,
    selector=dict(type="surface"),
)

fig = plot_coils(
    muse_coilset,
    fig=fig,
    color="#510076",
    lw=7,
    showgrid=False,
    showticklabels=False,
    showaxislabels=False,
    zeroline=False,
)

fig = plot_dipoles(
    muse_dipoles,
    fig=fig,
    threshold=0.5,
    size=2,
    opacity=0.85,
    color_positive="#D20000",
    color_negative="#004BE2",
    showgrid=False,
    showticklabels=False,
    showaxislabels=False,
    zeroline=False,
)

fig.update_layout(
    width=1200,
    height=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    paper_bgcolor="white",
    showlegend=False,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.8),
        ),
    ),
)

fig.show()

In [5]:
# continuous rho optimization from cold start (rho=0 for all dipoles)

coil_grid   = LinearGrid(N=40)
plasma_grid = LinearGrid(M=6, N=20, NFP=muse_eq.NFP, sym=muse_eq.sym)

weights = {
    "quadratic flux": 10000,
    "discreteness": 1,
    "volume": 1
}

muse_dipoles.rho = np.zeros(len(muse_dipoles))
optimizer = Optimizer("scipy-l-bfgs-b")

obj = ObjectiveFunction(
        (
            QuadraticFluxPM(
                muse_eq, field=muse_dipoles, eval_grid=plasma_grid, field_fixed=muse_coilset,
                vacuum=True, weight=weights["quadratic flux"], bs_chunk_size=50,
            ),
        )
     )

(muse_continuous_optimization,), _ = optimizer.optimize(
    muse_dipoles,
    objective=obj,
    maxiter=50,
    verbose=3,
    ftol=1e-08,
    copy=True,
)

Building objective: Quadratic flux
Precomputing transforms


/Users/jessicali/DESC/desc/utils.py:511: FutureWarning:

argument axis has been renamed to axis_or_vector, axis will be removed in a future release

/Users/jessicali/DESC/desc/utils.py:511: FutureWarning:

argument angle has been renamed to angle_or_vector, angle will be removed in a future release



Using precomputed induction matrix for 1 fixed-geometry dipole field(s)
Timer: Precomputing transforms = 18.5 sec
ObjectiveFunction deriv_mode : blocked
Quadratic flux jac_chunk_size: None
Timer: Objective build = 45.1 sec
Number of parameters: 75460
Number of objectives: 287
Timer: Initializing the optimization = 57.9 sec

Starting optimization
Using method: scipy-l-bfgs-b
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   


/Users/jessicali/DESC/desc/utils.py:563: FutureWarning:

constants is deprecated and will be removed in a future release. Users should not include constants in the arguments of their objective compute methods. Instead declare all the constants in the build method and use as self._constants.



       0              1          6.520e+07                                    5.249e+04   
       1              3          5.211e+07      1.309e+07      5.000e+00      4.464e+04   
       2              4          6.257e+06      4.585e+07      3.678e+01      7.330e+03   
       3              5          4.495e+06      1.761e+06      6.461e+00      6.966e+03   
       4              6          1.825e+06      2.670e+06      1.717e+01      3.641e+03   
       5              7          1.225e+06      6.005e+05      1.465e+01      9.730e+03   
       6              8          7.139e+05      5.107e+05      5.989e+00      2.296e+03   
       7              9          6.343e+05      7.956e+04      1.626e+00      1.587e+03   
       8             10          5.101e+05      1.243e+05      4.681e+00      1.303e+03   
       9             11          4.439e+05      6.615e+04      5.758e+00      4.771e+03   
      10             12          3.679e+05      7.602e+04      2.228e+00      1.577e+03   

In [ ]:
results = calculate_bn_error(
    eq=muse_eq,
    dipoles=muse_dipoles,
    coils=muse_coilset,
    M=64,
    N=64,
    plot=True,
)

In [ ]:
results = calculate_bn_error(
    eq=muse_eq,
    dipoles=muse_continuous_optimization,
    coils=muse_coilset,
    M=64,
    N=64,
    plot=True,
)

In [ ]:
rho_muse = plot_rho_distribution(
    muse_continuous_optimization,
    name="MUSE",
)

In [ ]:
obj2 = ObjectiveFunction(
        (
            QuadraticFluxPM(
                muse_eq, field=muse_continuous_optimization, eval_grid=plasma_grid, field_fixed=muse_coilset,
                vacuum=True, weight=weights["quadratic flux"], bs_chunk_size=50,
            ),
            DipoleDiscreteness(
                dipole=muse_continuous_optimization, weight=weights["discreteness"], 
            ),
            # DipoleVolume(
            #     dipole=muse_dipoles, weight=weights["volume"], 
            # )
        )
     )

(muse_discrete_optimization1,), _ = optimizer.optimize(
    muse_continuous_optimization,
    objective=obj2,
    maxiter=500,
    verbose=3,
    ftol=1e-08,
    copy=True,
)

In [ ]:
results = calculate_bn_error(
    eq=muse_eq,
    dipoles=muse_discrete_optimization1,
    coils=muse_coilset,
    M=64,
    N=64,
    plot=True,
)

In [ ]:
rho_muse = plot_rho_distribution(
    muse_discrete_optimization1,
    name="MUSE",
)

In [ ]:
obj3 = ObjectiveFunction(
        (
            QuadraticFluxPM(
                muse_eq, field=muse_discrete_optimization1, eval_grid=plasma_grid, field_fixed=muse_coilset,
                vacuum=True, weight=weights["quadratic flux"], bs_chunk_size=50,
            ),
            DipoleDiscreteness(
                dipole=muse_discrete_optimization1, weight=weights["discreteness"]*5, 
            ),
            # DipoleVolume(
            #     dipole=muse_dipoles, weight=weights["volume"], 
            # )
        )
     )

(muse_discrete_optimization2,), _ = optimizer.optimize(
    muse_discrete_optimization1,
    objective=obj3,
    maxiter=500,
    verbose=3,
    ftol=1e-08,
    copy=True,
)

In [ ]:
results = calculate_bn_error(
    eq=muse_eq,
    dipoles=muse_discrete_optimization2,
    coils=muse_coilset,
    M=64,
    N=64,
    plot=True,
)

In [ ]:
rho_muse = plot_rho_distribution(
    muse_discrete_optimization2,
    name="MUSE",
)

In [ ]:
obj4 = ObjectiveFunction(
        (
            QuadraticFluxPM(
                muse_eq, field=muse_discrete_optimization2, eval_grid=plasma_grid, field_fixed=muse_coilset,
                vacuum=True, weight=weights["quadratic flux"], bs_chunk_size=50,
            ),
            DipoleDiscreteness(
                dipole=muse_discrete_optimization2, weight=weights["discreteness"]*10, 
            ),
            # DipoleVolume(
            #     dipole=muse_dipoles, weight=weights["volume"], 
            # )
        )
     )

(muse_discrete_optimization3,), _ = optimizer.optimize(
    muse_discrete_optimization2,
    objective=obj4,
    maxiter=500,
    verbose=3,
    ftol=1e-08,
    copy=True,
)

In [ ]:
results = calculate_bn_error(
    eq=muse_eq,
    dipoles=muse_discrete_optimization3,
    coils=muse_coilset,
    M=64,
    N=64,
    plot=True,
)

In [ ]:
rho_muse = plot_rho_distribution(
    muse_discrete_optimization3,
    name="MUSE",
)

In [ ]:
import numpy as np
import jax.numpy as jnp

x0 = jnp.asarray(muse_continuous_optimization.rho)
xf = jnp.asarray(muse_discrete_optimization1.rho)

print("Maximum parameter change:",
      np.max(np.abs(np.asarray(xf - x0))))

print("Initial objective:", float(obj2.compute_scalar(x0)))
print("Final objective:  ", float(obj2.compute_scalar(xf)))

In [ ]:
obj2.print_value(
    x=xf,
    x0=x0,
    fse=None,
    f0se=None,
)